#Excercise


Replace MobileNetV2 with ResNet50 and Compare:

* Training time

* Accuracy





In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
import time

In [25]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

In [9]:
# 2: ResNet50

In [26]:
resnet = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

resnet.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Resizing(224, 224),
    tf.keras.layers.Lambda(
        tf.keras.applications.resnet50.preprocess_input
    ),
    resnet,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    validation_split=0.1,
    batch_size=32
)

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 137s 98ms/step - accuracy: 0.8873 - loss: 0.3323 - val_accuracy: 0.8990 - val_loss: 0.2877
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 136s 96ms/step - accuracy: 0.9023 - loss: 0.2855 - val_accuracy: 0.9076 - val_loss: 0.2694
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 135s 96ms/step - accuracy: 0.9137 - loss: 0.2519 - val_accuracy: 0.9052 - val_loss: 0.2606
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 135s 96ms/step - accuracy: 0.9210 - loss: 0.2289 - val_accuracy: 0.9108 - val_loss: 0.2509
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 143s 97ms/step - accuracy: 0.9286 - loss: 0.2089 - val_accuracy: 0.9174 - val_loss: 0.2354


Add data augmentation:

```
tf.keras.layers.RandomFlip("horizontal")
tf.keras.layers.RandomRotation(0.1)
```

In [29]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1)
])

resnet_aug = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
resnet_aug.trainable = False

model_aug = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    augmentation,
    tf.keras.layers.Resizing(224, 224),
    tf.keras.layers.Lambda(tf.keras.applications.resnet50.preprocess_input),
    resnet_aug,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation="softmax")
])

model_aug.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_aug = model_aug.fit(
    x_train, y_train,
    epochs=5,
    validation_split=0.1,
    batch_size=32
)

print(f"Without augmentation: {history.history['val_accuracy'][-1]:.2%}")
print(f"With augmentation: {history_aug.history['val_accuracy'][-1]:.2%}")

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 235s 162ms/step - accuracy: 0.6986 - loss: 0.8776 - val_accuracy: 0.8794 - val_loss: 0.3508
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 224s 159ms/step - accuracy: 0.7634 - loss: 0.6911 - val_accuracy: 0.8764 - val_loss: 0.3514
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 224s 159ms/step - accuracy: 0.7769 - loss: 0.6541 - val_accuracy: 0.8904 - val_loss: 0.3122
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 224s 159ms/step - accuracy: 0.7888 - loss: 0.6258 - val_accuracy: 0.8926 - val_loss: 0.3012
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 225s 160ms/step - accuracy: 0.7944 - loss: 0.6050 - val_accuracy: 0.9016 - val_loss: 0.2867
Without augmentation: 91.74%
With augmentation: 90.16%


Fine-tune only:

* Last 20 layers
* Last 50 layers

and Compare results.

In [30]:
# Save the trained baseline weights for a fair comparison
original_weights = model_aug.get_weights()

# Fine-tune the last 20 layers
resnet_aug.trainable = True
for layer in resnet_aug.layers[:-20]:
    layer.trainable = False
for layer in resnet_aug.layers[-20:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_20 = model_aug.fit(
    x_train, y_train,
    epochs=5,
    validation_split=0.1,
    batch_size=32
)

accuracy_20 = history_20.history["val_accuracy"][-1]

# Reset weights before trying the last 50 layers
model_aug.set_weights(original_weights)
resnet_aug.trainable = True
for layer in resnet_aug.layers[:-50]:
    layer.trainable = False
for layer in resnet_aug.layers[-50:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_50 = model_aug.fit(
    x_train, y_train,
    epochs=5,
    validation_split=0.1,
    batch_size=32
)

accuracy_50 = history_50.history["val_accuracy"][-1]

print(f"Last 20 layers: {accuracy_20:.2%}")
print(f"Last 50 layers: {accuracy_50:.2%}")

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 271s 187ms/step - accuracy: 0.8292 - loss: 0.5054 - val_accuracy: 0.9112 - val_loss: 0.2508
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 260s 185ms/step - accuracy: 0.8528 - loss: 0.4387 - val_accuracy: 0.9230 - val_loss: 0.2261
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 260s 185ms/step - accuracy: 0.8678 - loss: 0.4009 - val_accuracy: 0.9214 - val_loss: 0.2289
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 259s 184ms/step - accuracy: 0.8792 - loss: 0.3675 - val_accuracy: 0.9282 - val_loss: 0.2142
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 259s 184ms/step - accuracy: 0.8877 - loss: 0.3370 - val_accuracy: 0.9326 - val_loss: 0.2077
Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 324s 223ms/step - accuracy: 0.8453 - loss: 0.4704 - val_accuracy: 0.9210 - val_loss: 0.2272
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 310s 220ms/step - accuracy: 0.8756 - loss: 0.3821 - val_accuracy: 0.9378 - val_loss: 0.1979
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 310s 221ms/step - accuracy: 

In [ ]:
# after multiple sessions of trial and error -mostly error- I managed to get the accuracy from 0.09 -initial notebook that was deleted -to 0.94 now
# x_train and x_test / 225.0 was to balme for earlier poor results.